In [0]:
dbutils.widgets.text("catalog", "anurag_dev")
dbutils.widgets.text("src_schema", "bronze")
dbutils.widgets.text("tgt_schema", "silver")

catalog = dbutils.widgets.get("catalog")
src = dbutils.widgets.get("src_schema")
tgt = dbutils.widgets.get("tgt_schema")

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ---- CUSTOMERS (dedup by customer_id, keep latest updated_at) ----
customers_df = spark.table(f"{catalog}.{src}.customers")
w = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())
customers_clean = customers_df.withColumn("rn", row_number().over(w)) \
    .filter("rn = 1").drop("rn", "ingestion_timestamp", "source_file_name")

# UPSERT into silver
if spark.catalog.tableExists(f"{catalog}.{tgt}.customers"):
    dt = DeltaTable.forName(spark, f"{catalog}.{tgt}.customers")
    dt.alias("tgt").merge(
        customers_clean.alias("src"),
        "tgt.customer_id = src.customer_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    customers_clean.write.format("delta").saveAsTable(f"{catalog}.{tgt}.customers")

# ---- ORDERS joined with customers ----
orders_df = spark.table(f"{catalog}.{src}.orders")
orders_with_cust = orders_df.join(
    spark.table(f"{catalog}.{tgt}.customers").select("customer_id","name","city","state"),
    on="customer_id", how="left"
)
orders_with_cust.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.{tgt}.orders_enriched")

print("✅ Silver processing complete")